## Urbanride_02 — Bronze Events Ingestion
**Method:** Auto Loader (`cloudFiles`) — incremental, checkpoint-based  
**Inputs:** `/Volumes/urbanride/bronze/raw_files/` — trips, clickstream  
**Outputs:** `urbanride.bronze.trips` · `urbanride.bronze.clickstream`  
**Schedule:** Daily · 01:00 AM  
**Trigger:** `availableNow=True` — processes all new files then stops

**Why Auto Loader over COPY INTO for JSON:**
- Schema can evolve — new fields in trips/clickstream handled automatically
- Checkpoint guarantees exactly-once ingestion — no duplicates on retry
- Streaming-ready foundation — Silver can read these tables as a stream later

In [0]:
import time
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, BooleanType, TimestampType, IntegerType
)

# CATALOG CONFIG 
CATALOG   = 'urbanride'
BRONZE    = f'{CATALOG}.bronze'
BASE_PATH = f'/Volumes/{CATALOG}/bronze/raw_files'

# SOURCE PATHS 
TRIPS_PATH       = f'{BASE_PATH}/trips/'
CLICKSTREAM_PATH = f'{BASE_PATH}/clickstream/'

#  CHECKPOINT PATHS 
# Checkpoint stores which files Auto Loader has already processed
# Must be a stable path — never delete these while the pipeline is active
TRIPS_CHECKPOINT       = f'{BASE_PATH}/_checkpoints/trips'
CLICKSTREAM_CHECKPOINT = f'{BASE_PATH}/_checkpoints/clickstream'

# SCHEMA LOCATION 
# Auto Loader stores inferred/evolved schema here
# Even with schema hints, this path is required
TRIPS_SCHEMA_LOC       = f'{BASE_PATH}/_schema/trips'
CLICKSTREAM_SCHEMA_LOC = f'{BASE_PATH}/_schema/clickstream'

spark.sql(f'USE CATALOG {CATALOG}')

print(f'Catalog    : {CATALOG}')
print(f'Bronze     : {BRONZE}')
print(f'Trips src  : {TRIPS_PATH}')
print(f'Events src : {CLICKSTREAM_PATH}')
print(f'Checkpoints: {BASE_PATH}/_checkpoints/')
print('Config ready.')


Catalog    : urbanride
Bronze     : urbanride.bronze
Trips src  : /Volumes/urbanride/bronze/raw_files/trips/
Events src : /Volumes/urbanride/bronze/raw_files/clickstream/
Checkpoints: /Volumes/urbanride/bronze/raw_files/_checkpoints/
Config ready.


In [0]:
# Explicit schema hints — faster than inference, more reliable at scale
# Auto Loader uses these as the base schema
# If a new field appears in future files — schemaEvolutionMode handles it

# --------------------- TRIPS SCHEMA -------------------------
trips_schema = StructType([
    StructField('trip_id',          StringType(),    True),
    StructField('customer_id',      StringType(),    True),
    StructField('driver_id',        StringType(),    True),
    StructField('city',             StringType(),    True),
    StructField('pickup_time',      StringType(),    True),  # cast to timestamp in Silver
    StructField('drop_time',        StringType(),    True),  # cast to timestamp in Silver
    StructField('distance_km',      DoubleType(),    True),
    StructField('fare_amount',      DoubleType(),    True),
    StructField('surge_multiplier', DoubleType(),    True),
    StructField('trip_status',      StringType(),    True),
    StructField('vehicle_type',     StringType(),    True),
    StructField('weather',          StringType(),    True),
    StructField('is_ghost_trip',    BooleanType(),   True),
    StructField('is_fare_inflated', BooleanType(),   True),
    StructField('day_type',         StringType(),    True),
])

# --------------------- CLICKSTREAM SCHEMA -------------------------
clickstream_schema = StructType([
    StructField('event_id',        StringType(),  True),
    StructField('customer_id',     StringType(),  True),
    StructField('session_id',      StringType(),  True),
    StructField('event_type',      StringType(),  True),
    StructField('event_timestamp', StringType(),  True),  # cast to timestamp in Silver
    StructField('city',            StringType(),  True),
    StructField('device_type',     StringType(),  True),
    StructField('ab_test_group',   StringType(),  True),
    StructField('ab_test_name',    StringType(),  True),
])

print('Schema hints defined.')
print(f'  Trips schema       : {len(trips_schema.fields)} fields')
print(f'  Clickstream schema : {len(clickstream_schema.fields)} fields')
print()
print('Note: pickup_time, drop_time, event_timestamp kept as STRING in Bronze')
print('Timestamp casting happens in Silver — Bronze stores raw values')


Schema hints defined.
  Trips schema       : 15 fields
  Clickstream schema : 9 fields

Note: pickup_time, drop_time, event_timestamp kept as STRING in Bronze
Timestamp casting happens in Silver — Bronze stores raw values


In [0]:
# Run this once before Cell 3
# spark.sql(f'DROP TABLE IF EXISTS {BRONZE}.trips')
# spark.sql(f'DROP TABLE IF EXISTS {BRONZE}.clickstream')
# dbutils.fs.rm(TRIPS_CHECKPOINT, recurse=True)
# dbutils.fs.rm(CLICKSTREAM_CHECKPOINT, recurse=True)
# dbutils.fs.rm(TRIPS_SCHEMA_LOC, recurse=True)
# dbutils.fs.rm(CLICKSTREAM_SCHEMA_LOC, recurse=True)

False

In [0]:
# Create target Delta tables before Auto Loader writes to them
# IF NOT EXISTS — safe to re-run
print('Creating Bronze Delta tables if not exist...')

# ─------------------------- TRIPS -------------------------

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.trips (
        trip_id          STRING,
        customer_id      STRING,
        driver_id        STRING,
        city             STRING,
        pickup_time      STRING,
        drop_time        STRING,
        distance_km      DOUBLE,
        fare_amount      DOUBLE,
        surge_multiplier DOUBLE,
        trip_status      STRING,
        vehicle_type     STRING,
        weather          STRING,
        is_ghost_trip    BOOLEAN,
        is_fare_inflated BOOLEAN,
        day_type         STRING,
        _ingested_at     TIMESTAMP
    )
    USING DELTA
    COMMENT 'Bronze raw trip event records — append only'
""")
print('urbanride.bronze.trips — ready')

# ---------------------- CLICKSTREAM -------------------------

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.clickstream (
        event_id        STRING,
        customer_id     STRING,
        session_id      STRING,
        event_type      STRING,
        event_timestamp STRING,
        city            STRING,
        device_type     STRING,
        ab_test_group   STRING,
        ab_test_name    STRING,
        _ingested_at    TIMESTAMP
    )
    USING DELTA
    COMMENT 'Bronze raw clickstream event records — append only'
""")
print('Urbanride.bronze.clickstream — ready')
print()
print('All Bronze event tables ready for Auto Loader.')


Creating Bronze Delta tables if not exist...
zipcab.bronze.trips — ready
zipcab.bronze.clickstream — ready

All Bronze event tables ready for Auto Loader.


In [0]:
# Auto Loader — incremental JSON ingestion for trips
# availableNow=True — processes all new files then stops (batch-like behavior)
# Checkpoint remembers exactly which files were loaded — safe to re-run daily

from pyspark.sql.functions import current_timestamp

print('Loading trips via Auto Loader...')
t0 = time.time()

(
    spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', TRIPS_SCHEMA_LOC)
        .schema(trips_schema)
        .load(TRIPS_PATH)
        .withColumn('_ingested_at', current_timestamp())
    .writeStream
        .format('delta')
        .option('checkpointLocation', TRIPS_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(f'{BRONZE}.trips')
        .awaitTermination()
)

n = spark.table(f'{BRONZE}.trips').count()
print(f'  Rows in bronze.trips : {n:,}')
print(f'  Time                 : {round(time.time()-t0, 2)}s')
print('  Trips loaded.')

Loading trips via Auto Loader...
  Rows in bronze.trips : 21,359,745
  Time                 : 41.27s
  Trips loaded.


In [0]:
# Auto Loader — incremental JSON ingestion for clickstream
print('Loading clickstream via Auto Loader...')
t0 = time.time()

from pyspark.sql.functions import current_timestamp

(
    spark.readStream
        .format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', CLICKSTREAM_SCHEMA_LOC)
        .schema(clickstream_schema)
        .load(CLICKSTREAM_PATH)
        .withColumn('_ingested_at', current_timestamp())
    .writeStream
        .format('delta')
        .option('checkpointLocation', CLICKSTREAM_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(f'{BRONZE}.clickstream')
        .awaitTermination()
)

n = spark.table(f'{BRONZE}.clickstream').count()
print(f'Rows in bronze.clickstream : {n:,}')
print(f'Time : {round(time.time()-t0,2)}s')
print('Clickstream loaded.')


Loading clickstream via Auto Loader...


26/03/08 14:08:14 Spark Server has not sent updates for Streaming Query def59f9e-0397-4ae8-a324-8f895342ee8f in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 57954eff-7899-4011-ba5a-3488346a2704. This is typically not a problem.


Rows in bronze.clickstream : 24,407,534
Time : 71.41s
Clickstream loaded.


In [0]:
print('=== BRONZE EVENTS INGESTION VERIFICATION ===')
print()

tables = [
    (f'{BRONZE}.trips','Trips'),
    (f'{BRONZE}.clickstream', 'Clickstream'),
]

print(f'  {"Table":<35} {"Rows":>15}  {"Columns":>10}')
print('  ' + '-'*65)
for tbl, name in tables:
    df = spark.table(tbl)
    n  = df.count()
    cols = len(df.columns)
    print(f'  {tbl:<35} {n:>15,}  {cols:>10}')

print()

# Checkpoint confirmation — if checkpoint exists, Auto Loader ran successfully
print('Checkpoint status:')
for name, chk in [('Trips', TRIPS_CHECKPOINT), ('Clickstream', CLICKSTREAM_CHECKPOINT)]:
    try:
        files = dbutils.fs.ls(chk)
        print(f'  {name:<15}: checkpoint exists ({len(files)} files) — incremental runs active')
    except:
        print(f'  {name:<15}: checkpoint not found — check if Auto Loader ran')

print()

# Sample data
print('Sample from bronze.trips:')
spark.table(f'{BRONZE}.trips') \
    .select('trip_id','city','trip_status','fare_amount','weather','day_type','_ingested_at') \
    .limit(3).show(truncate=False)

print('Sample from bronze.clickstream:')
spark.table(f'{BRONZE}.clickstream') \
    .select('event_id','customer_id','event_type','ab_test_group','_ingested_at') \
    .limit(3).show(truncate=False)


=== BRONZE EVENTS INGESTION VERIFICATION ===

  Table                                          Rows     Columns
  -----------------------------------------------------------------
  urbanride.bronze.trips                   21,359,745          16
  urbanride.bronze.clickstream             24,407,534          10

Checkpoint status:
  Trips          : checkpoint exists (4 files) — incremental runs active
  Clickstream    : checkpoint exists (4 files) — incremental runs active

Sample from bronze.trips:
+------------------------------------+------+-----------+-----------+-------+--------+-----------------------+
|trip_id                             |city  |trip_status|fare_amount|weather|day_type|_ingested_at           |
+------------------------------------+------+-----------+-----------+-------+--------+-----------------------+
|9423e154-0fad-41e5-bf00-4851f29be23c|Delhi |completed  |56.99      |Clear  |holiday |2026-03-08 14:05:47.681|
|17071f9a-37d1-49af-8f76-0c9269882c50|Pune  |comple

In [0]:
# Final summary across all 5 Bronze tables — ZipCab_01 + ZipCab_02
print('=== Urbanride BRONZE LAYER — COMPLETE ===')
print()

all_tables = [
    (f'{BRONZE}.customers',   'COPY INTO',   'CSV'),
    (f'{BRONZE}.drivers',     'COPY INTO',   'CSV'),
    (f'{BRONZE}.payments',    'COPY INTO',   'CSV'),
    (f'{BRONZE}.trips',       'Auto Loader', 'JSON'),
    (f'{BRONZE}.clickstream', 'Auto Loader', 'JSON'),
]

print(f'  {"Table":<35} {"Method":<15} {"Format":<8} {"Rows":>15}')
print('  ' + '-'*80)
total = 0
for tbl, method, fmt in all_tables:
    try:
        n = spark.table(tbl).count()
        total += n
        print(f'  {tbl:<35} {method:<15} {fmt:<8} {n:>15,}')
    except Exception as e:
        print(f'  {tbl:<35} {"ERROR":<15} {fmt:<8} {str(e):>15}')

print('  ' + '-'*80)
print(f'  {"TOTAL":<35} {"":<15} {"":<8} {total:>15,}')
print()
print('Bronze layer complete. Next: Silver transformation notebooks.')
print('ZipCab_03 — Silver Customers')
print('ZipCab_04 — Silver Rides')
print('ZipCab_05 — Silver Payments')
print('ZipCab_06 — Silver Clickstream')


=== ZIPCAB BRONZE LAYER — COMPLETE ===

  Table                               Method          Format              Rows
  --------------------------------------------------------------------------------
  urbanride.bronze.customers          COPY INTO       CSV               89,963
  urbanride.bronze.drivers            COPY INTO       CSV               20,000
  urbanride.bronze.payments           COPY INTO       CSV           19,600,753
  urbanride.bronze.trips              Auto Loader     JSON          21,359,745
  urbanride.bronze.clickstream        Auto Loader     JSON          24,407,534
  --------------------------------------------------------------------------------
  TOTAL                                                             65,477,995

Bronze layer complete. Next: Silver transformation notebooks.
ZipCab_03 — Silver Customers
ZipCab_04 — Silver Rides
ZipCab_05 — Silver Payments
ZipCab_06 — Silver Clickstream
